# Fake Review Classification — Improved DNN, ResDNN, Wide&Deep

Notebook ini menambahkan:
- DNN (BatchNorm + Dropout)
- ResDNN (Residual connections)
- Wide&Deep (TF-IDF + fitur linguistik sederhana)

Pastikan file **dataset_final.csv** tersedia di working directory.


In [1]:
# Install dependencies (jalankan sekali)
!pip -q install -U pandas numpy scikit-learn matplotlib seaborn torch tqdm


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report    

from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cpu


## 1) Load dataset_final.csv
Kolom yang digunakan: `text_final` dan `label_final`.


In [3]:
df = pd.read_csv('dataset_final.csv')

# Helper: konversi label teks/angka -> 0/1
def normalize_label_series(s: pd.Series) -> pd.Series:
    s = s.copy()

    # jika sudah numeric, langsung pakai (isi NaN jadi 0)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(int)

    # kalau object/string: map Real/Fake atau variasinya
    s2 = s.astype(str).str.strip().str.lower()

    mapping = {
        "real": 0,
        "fake": 1,
        "0": 0,
        "1": 1,
        "true": 1,
        "false": 0,
    }

    s2 = s2.map(mapping)

    # kalau masih ada NaN (nilai aneh), fallback jadi 0
    return s2.fillna(0).astype(int)

# Normalisasi kolom label yang mungkin ada
for col in ['Label', 'label_final', 'decision']:
    if col in df.columns:
        df[col] = normalize_label_series(df[col])

# Tentukan kolom teks dan label final (sesuai dataset kamu)
# default: text_final + label_final
text_col = 'text_final'
label_col = 'label_final'

if text_col not in df.columns:
    raise ValueError(f"Kolom teks '{text_col}' tidak ditemukan. Kolom yang ada: {list(df.columns)}")

if label_col not in df.columns:
    # fallback: kalau label_final tidak ada, coba 'Label'
    if 'Label' in df.columns:
        label_col = 'Label'
    else:
        raise ValueError(f"Kolom label '{label_col}' tidak ditemukan. Kolom yang ada: {list(df.columns)}")

df = df[[text_col, label_col]].copy()
df.columns = ['text', 'label']
df = df.dropna()

print(f"Dataset shape: {df.shape}")
print("Label distribution:\n", df['label'].value_counts())
df.head()


Dataset shape: (7875, 2)
Label distribution:
 label
0    6533
1    1342
Name: count, dtype: int64


,text,label
0,barang pesanan datang sesuai jadwal barang pes...,0
1,bahan: bagus manfaat produk: sangat baik jenis...,0
2,bahan: bagus manfaat produk: mencerahkan jenis...,0
3,bahan: sesuai pemesanan manfaat produk: mencer...,0
4,bahan: bagus sdah berlangganan manfaat produk:...,0


## 2) Train/Val/Test split

In [4]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


Train: 5512 | Val: 1181 | Test: 1182


## 3) TF-IDF Vectorization

In [5]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = tfidf.fit_transform(train_df['text']).toarray()
X_val_tfidf = tfidf.transform(val_df['text']).toarray()
X_test_tfidf = tfidf.transform(test_df['text']).toarray()

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

print('TF-IDF shape:', X_train_tfidf.shape)


TF-IDF shape: (5512, 5000)


## 4) Wide features (untuk Wide&Deep)

In [6]:
def extract_wide_features(df_in, text_column='text'):
    features = pd.DataFrame()
    s = df_in[text_column].astype(str)

    features['review_length'] = s.str.len()
    features['word_count'] = s.str.split().str.len()
    features['exclamation_count'] = s.str.count('!')
    features['question_count'] = s.str.count(r'\?')

    def caps_ratio(x):
        x = str(x)
        return (sum(1 for c in x if c.isupper()) / len(x)) if len(x) > 0 else 0

    features['caps_ratio'] = s.apply(caps_ratio)

    # Emoji heuristic
    features['emoji_count'] = s.apply(lambda x: sum(1 for c in x if ord(c) > 127000))

    # Repeated word ratio
    def repeated_ratio(x):
        words = str(x).split()
        if len(words) == 0:
            return 0
        rep = [w for w in words if words.count(w) > 1]
        return len(rep) / len(words)

    features['repeated_words'] = s.apply(repeated_ratio)
    return features

train_wide_df = extract_wide_features(train_df)
val_wide_df = extract_wide_features(val_df)
test_wide_df = extract_wide_features(test_df)

scaler_wide = StandardScaler()
X_train_wide = scaler_wide.fit_transform(train_wide_df)
X_val_wide = scaler_wide.transform(val_wide_df)
X_test_wide = scaler_wide.transform(test_wide_df)

print('Wide shape:', X_train_wide.shape)
train_wide_df.head()


Wide shape: (5512, 7)


,review_length,word_count,exclamation_count,question_count,caps_ratio,emoji_count,repeated_words
0,13,2,0,0,0.0,0,0.000000
1,56,8,0,0,0.0,0,0.000000
2,167,27,0,0,0.0,2,0.148148
3,173,26,0,0,0.0,0,0.076923
4,82,14,0,0,0.0,3,0.000000


## 5) Model definitions

In [7]:
class DNN(nn.Module):
    def __init__(self, input_size, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


class ResDNN(nn.Module):
    def __init__(self, input_size, num_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(input_size, 512)

        self.fc1 = nn.Linear(512, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 512)
        self.bn2 = nn.BatchNorm1d(512)

        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = nn.Linear(256, 256)
        self.bn4 = nn.BatchNorm1d(256)

        self.fc5 = nn.Linear(256, 128)
        self.bn5 = nn.BatchNorm1d(128)
        self.fc_out = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        x = F.relu(self.input_proj(x))
        x = self.dropout(x)

        identity = x
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        out = out + identity
        out = F.relu(out)
        out = self.dropout(out)

        identity2 = F.relu(self.bn3(self.fc3(out)))
        out = self.dropout(identity2)
        out = self.bn4(self.fc4(out))
        out = out + identity2
        out = F.relu(out)
        out = self.dropout(out)

        out = F.relu(self.bn5(self.fc5(out)))
        out = self.dropout(out)
        return self.fc_out(out)


class WideDeep(nn.Module):
    def __init__(self, text_input_size, wide_input_size, num_classes=2):
        super().__init__()
        self.deep = nn.Sequential(
            nn.Linear(text_input_size, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3)
        )
        self.wide = nn.Sequential(
            nn.Linear(wide_input_size, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU()
        )
        self.fc_out = nn.Sequential(
            nn.Linear(128 + 32, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes)
        )

    def forward(self, x_text, x_wide):
        deep_feat = self.deep(x_text)
        wide_feat = self.wide(x_wide)
        combined = torch.cat([deep_feat, wide_feat], dim=1)
        return self.fc_out(combined)


class WideDeepDataset(Dataset):
    def __init__(self, X_text, X_wide, y):
        self.X_text = torch.FloatTensor(X_text)
        self.X_wide = torch.FloatTensor(X_wide)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_text[idx], self.X_wide[idx], self.y[idx]

print('✓ Models ready')


✓ Models ready


## 6) Training & evaluation

In [8]:
def evaluate(
    model,
    test_loader,
    device=device,
    name='Model',
    is_widedeep=False
):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Evaluating {name}"):
            if is_widedeep:
                X_text, X_wide, y = batch
                logits = model(X_text.to(device), X_wide.to(device))
            else:
                X, y = batch
                logits = model(X.to(device))

            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labels.extend(y.numpy())

    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0, 1])
    cm = confusion_matrix(labels, preds)

    print("\n" + "=" * 60)
    print(f"{name} (test)")
    print("=" * 60)
    print(f"Accuracy: {acc:.4f}")
    print(f"Class 0 (Real): P={p[0]:.4f}, R={r[0]:.4f}, F1={f1[0]:.4f}")
    print(f"Class 1 (Fake): P={p[1]:.4f}, R={r[1]:.4f}, F1={f1[1]:.4f}")
    print("Confusion Matrix:\n", cm)

    print(
        classification_report(
            labels,
            preds,
            target_names=["Real", "Fake"],
            digits=4,
            zero_division=0
        )
    )
    print("=" * 60)

    rep = classification_report(
        labels,
        preds,
        target_names=["Real", "Fake"],
        output_dict=True,
        zero_division=0
    )

    return {
        "accuracy": rep["accuracy"],
        "precision_fake": rep["Fake"]["precision"],
        "recall_fake": rep["Fake"]["recall"],
        "f1_fake": rep["Fake"]["f1-score"],
        "confusion_matrix": cm
    }


In [9]:
def eval_loader_widedeep(model, loader, device=device):
    model.eval()
    preds, labels = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for X_text, X_wide, y in loader:
            X_text = X_text.to(device)
            X_wide = X_wide.to(device)
            y = y.to(device)

            logits = model(X_text, X_wide)
            loss = criterion(logits, y)
            total_loss += loss.item()

            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labels.extend(y.cpu().numpy())

    val_loss = total_loss / max(1, len(loader))
    acc = accuracy_score(labels, preds)

    # metrik fokus kelas Fake=1
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0, 1])
    precision_fake, recall_fake, f1_fake = p[1], r[1], f1[1]

    return val_loss, acc, precision_fake, recall_fake, f1_fake

In [10]:
def eval_loader_single_input(model, loader, device=device):
    model.eval()
    preds, labels = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = criterion(logits, y)
            total_loss += loss.item()

            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labels.extend(y.cpu().numpy())

    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0,1])
    return total_loss / max(1, len(loader)), acc, p[1], r[1], f1[1]  # fokus Fake=1


In [11]:
def train_model_with_table(model, train_loader, val_loader, num_epochs=3, lr=1e-3, device=device, model_name="Model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    rows = []
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, num_epochs+1):
        model.train()
        total_train_loss = 0.0

        for X, y in tqdm(train_loader, desc=f"[{model_name}] Epoch {epoch}/{num_epochs}"):
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        train_loss = total_train_loss / max(1, len(train_loader))
        val_loss, acc, prec_fake, rec_fake, f1_fake = eval_loader_single_input(model, val_loader, device=device)

        rows.append({
            "Epoch": epoch,
            "Training Loss": round(train_loss, 6),
            "Validation Loss": round(val_loss, 6),
            "Accuracy": round(acc, 6),
            "Precision Fake": round(prec_fake, 6),
            "Recall Fake": round(rec_fake, 6),
            "F1 Fake": round(f1_fake, 6),
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    hist_df = pd.DataFrame(rows)
    display(hist_df)
    return model, hist_df


In [12]:
def train_widedeep_with_table(
    model,
    train_loader,
    val_loader,
    num_epochs=3,
    lr=1e-3,
    device=device,
    model_name="Wide&Deep",
    use_scheduler=True
):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2
        )

    rows = []
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, num_epochs + 1):
        # ---- TRAIN ----
        model.train()
        total_train_loss = 0.0

        pbar = tqdm(train_loader, desc=f"[{model_name}] Epoch {epoch}/{num_epochs}")
        for X_text, X_wide, y in pbar:
            X_text = X_text.to(device)
            X_wide = X_wide.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(X_text, X_wide)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss = total_train_loss / max(1, len(train_loader))

        # ---- VALIDATION ----
        val_loss, acc, prec_fake, rec_fake, f1_fake = eval_loader_widedeep(
            model, val_loader, device=device
        )

        if scheduler is not None:
            scheduler.step(val_loss)

        rows.append({
            "Epoch": epoch,
            "Training Loss": round(train_loss, 6),
            "Validation Loss": round(val_loss, 6),
            "Accuracy": round(acc, 6),
            "Precision Fake": round(prec_fake, 6),
            "Recall Fake": round(rec_fake, 6),
            "F1 Fake": round(f1_fake, 6),
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    hist_df = pd.DataFrame(rows)
    display(hist_df)
    return model, hist_df


## 7) Run training (DNN, ResDNN, Wide&Deep)

In [13]:
# DataLoaders (DNN / ResDNN)
train_ds = TensorDataset(torch.FloatTensor(X_train_tfidf), torch.LongTensor(y_train))
val_ds = TensorDataset(torch.FloatTensor(X_val_tfidf), torch.LongTensor(y_val))
test_ds = TensorDataset(torch.FloatTensor(X_test_tfidf), torch.LongTensor(y_test))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

# DNN (train -> tabel epoch)
model_dnn = DNN(X_train_tfidf.shape[1])
model_dnn, hist_dnn = train_model_with_table(
    model_dnn, train_loader, val_loader,
    num_epochs=10, lr=1e-3, model_name='DNN'
)

# Test evaluation (dict output)
metrics_dnn = evaluate(model_dnn, test_loader, name='DNN', is_widedeep=False)

# ResDNN (train -> tabel epoch)
model_resdnn = ResDNN(X_train_tfidf.shape[1])
model_resdnn, hist_resdnn = train_model_with_table(
    model_resdnn, train_loader, val_loader,
    num_epochs=10, lr=1e-3, model_name='ResDNN'
)

# Test evaluation (dict output)
metrics_resdnn = evaluate(model_resdnn, test_loader, name='ResDNN', is_widedeep=False)

# Wide&Deep loaders
train_wd_ds = WideDeepDataset(X_train_tfidf, X_train_wide, y_train)
val_wd_ds = WideDeepDataset(X_val_tfidf, X_val_wide, y_val)
test_wd_ds = WideDeepDataset(X_test_tfidf, X_test_wide, y_test)

train_wd_loader = DataLoader(train_wd_ds, batch_size=32, shuffle=True)
val_wd_loader = DataLoader(val_wd_ds, batch_size=64, shuffle=False)
test_wd_loader = DataLoader(test_wd_ds, batch_size=64, shuffle=False)

# Wide&Deep (train -> tabel epoch)
model_wd = WideDeep(X_train_tfidf.shape[1], X_train_wide.shape[1])
model_wd, hist_wd = train_widedeep_with_table(
    model_wd, train_wd_loader, val_wd_loader,
    num_epochs=10, lr=1e-3, device=device
)

# Test evaluation (dict output)
metrics_wd = evaluate(model_wd, test_wd_loader, name='Wide&Deep', is_widedeep=True)

# Summary (consistent dict keys)
comp_df = pd.DataFrame({
    'Model': ['DNN', 'ResDNN', 'Wide&Deep'],
    'Accuracy': [metrics_dnn['accuracy'], metrics_resdnn['accuracy'], metrics_wd['accuracy']],
    'F1 (Fake)': [metrics_dnn['f1_fake'], metrics_resdnn['f1_fake'], metrics_wd['f1_fake']],
    'Precision (Fake)': [metrics_dnn['precision_fake'], metrics_resdnn['precision_fake'], metrics_wd['precision_fake']],
    'Recall (Fake)': [metrics_dnn['recall_fake'], metrics_resdnn['recall_fake'], metrics_wd['recall_fake']],
})

print('' + '='*70)
print('MODEL COMPARISON')
print('='*70)
print(comp_df.to_string(index=False))
print('='*70)

comp_df


[DNN] Epoch 10/10: 100%|██████████| 173/173 [00:04<00:00, 36.18it/s]


,Epoch,Training Loss,Validation Loss,Accuracy,Precision Fake,Recall Fake,F1 Fake
0,1,0.552672,0.363672,0.883150,0.769231,0.447761,0.566038
1,2,0.342919,0.332706,0.875529,0.759615,0.393035,0.518033
2,3,0.248199,0.358120,0.867062,0.707547,0.373134,0.488599
3,4,0.183476,0.400982,0.867062,0.683333,0.407960,0.510903
4,5,0.134670,0.467294,0.861135,0.635036,0.432836,0.514793
5,6,0.107671,0.472460,0.861981,0.633803,0.447761,0.524781
6,7,0.083366,0.521426,0.856901,0.606667,0.452736,0.518519
7,8,0.078546,0.567572,0.866215,0.686957,0.393035,0.500000
8,9,0.072523,0.563512,0.861981,0.679245,0.358209,0.469055
9,10,0.064649,0.570303,0.865368,0.643836,0.467662,0.541787


Evaluating DNN: 100%|██████████| 19/19 [00:00<00:00, 285.44it/s]



DNN (test)
Accuracy: 0.8748
Class 0 (Real): P=0.8888, R=0.9704, F1=0.9278
Class 1 (Fake): P=0.7411, R=0.4109, F1=0.5287
Confusion Matrix:
 [[951  29]
 [119  83]]
              precision    recall  f1-score   support

        Real     0.8888    0.9704    0.9278       980
        Fake     0.7411    0.4109    0.5287       202

    accuracy                         0.8748      1182
   macro avg     0.8149    0.6906    0.7282      1182
weighted avg     0.8635    0.8748    0.8596      1182



[ResDNN] Epoch 10/10: 100%|██████████| 173/173 [00:05<00:00, 30.49it/s]


...

[Wide&Deep] Epoch 10/10: 100%|██████████| 173/173 [00:04<00:00, 35.27it/s, loss=0.0016]


,Epoch,Training Loss,Validation Loss,Accuracy,Precision Fake,Recall Fake,F1 Fake
0,1,0.407032,0.345481,0.876376,0.720000,0.447761,0.552147
1,2,0.283676,0.351970,0.874682,0.827160,0.333333,0.475177
2,3,0.196866,0.408163,0.867062,0.729167,0.348259,0.471380
3,4,0.141283,0.479868,0.862828,0.648855,0.422886,0.512048
4,5,0.093128,0.539701,0.858594,0.608974,0.472637,0.532213
5,6,0.063310,0.647733,0.858594,0.630769,0.407960,0.495468
6,7,0.065115,0.647704,0.838273,0.526316,0.497512,0.511509
7,8,0.046095,0.695686,0.854361,0.596026,0.447761,0.511364
8,9,0.037269,0.768599,0.865368,0.664062,0.422886,0.516717
9,10,0.033125,0.741912,0.848434,0.562500,0.492537,0.525199


Evaluating Wide&Deep: 100%|██████████| 19/19 [00:00<00:00, 273.00it/s]



Wide&Deep (test)
Accuracy: 0.8790
Class 0 (Real): P=0.8974, R=0.9643, F1=0.9297
Class 1 (Fake): P=0.7287, R=0.4653, F1=0.5680
Confusion Matrix:
 [[945  35]
 [108  94]]
              precision    recall  f1-score   support

        Real     0.8974    0.9643    0.9297       980
        Fake     0.7287    0.4653    0.5680       202

    accuracy                         0.8790      1182
   macro avg     0.8131    0.7148    0.7488      1182
weighted avg     0.8686    0.8790    0.8678      1182

MODEL COMPARISON
    Model  Accuracy  F1 (Fake)  Precision (Fake)  Recall (Fake)
      DNN  0.874788   0.528662          0.741071       0.410891
   ResDNN  0.871404   0.482993          0.771739       0.351485
Wide&Deep  0.879019   0.567976          0.728682       0.465347


,Model,Accuracy,F1 (Fake),Precision (Fake),Recall (Fake)
0,DNN,0.874788,0.528662,0.741071,0.410891
1,ResDNN,0.871404,0.482993,0.771739,0.351485
2,Wide&Deep,0.879019,0.567976,0.728682,0.465347
